# LLaMA 3.1 8B — Full Watermarking Pipeline

Runs the complete experiment pipeline for `meta-llama/Llama-3.1-8B-Instruct`.

**Prerequisites:**
- A100 or T4 GPU runtime (Runtime → Change runtime type → GPU)
- HuggingFace account with LLaMA license accepted at huggingface.co/meta-llama/Llama-3.1-8B-Instruct
- HuggingFace access token

**Runtime estimate:** ~6-8 hours total on T4, ~4-5 hours on A100.

## Setup

Paste your HuggingFace token below, then run this cell.

In [1]:
import os, sys

HF_TOKEN   = ''   # paste your hf_... token here
GITHUB_URL = 'https://github.com/AliHasan-786/llm-invisible-watermarking.git'
BRANCH     = 'AmmarChanges'
REPO_DIR   = '/content/llm-invisible-watermarking'   # absolute path — safe to re-run
MODEL      = 'meta-llama/Llama-3.1-8B-Instruct'

if not os.path.exists(REPO_DIR):
    print(f'Cloning {BRANCH}...')
    os.system(f'git clone --branch {BRANCH} {GITHUB_URL} {REPO_DIR}')
else:
    print('Pulling latest...')
    os.system(f'cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

print('Installing packages...')
os.system('pip install -q transformers datasets accelerate scipy numpy matplotlib tqdm sentencepiece protobuf huggingface_hub')
print('✓ Packages installed')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ Logged into HuggingFace')
else:
    print('⚠️  HF_TOKEN is empty — paste your token above and re-run this cell')

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {name}  ({vram:.1f} GB VRAM)')
    if vram < 14:
        print('  ⚠️  <14 GB VRAM — LLaMA 8B needs ~14 GB fp16. Switch to A100.')
else:
    print('✗ No GPU — switch to a GPU runtime')

Cloning AmmarChanges...
Working dir: /content/llm-invisible-watermarking
Installing packages...
✓ Packages installed


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Logged into HuggingFace
✓ GPU: NVIDIA A100-SXM4-40GB  (42.4 GB VRAM)


In [ ]:
import shutil, glob

# Mount Google Drive — results saved here survive disconnects.
# Re-run this cell on reconnect to restore previous progress automatically.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS = '/content/drive/MyDrive/watermark_llama_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs('results', exist_ok=True)

def checkpoint():
    saved = []
    for fp in glob.glob('results/*llama*') + glob.glob('results/corpus_llama*'):
        shutil.copy2(fp, DRIVE_RESULTS)
        saved.append(os.path.basename(fp))
    if saved:
        print(f'✓ Checkpointed {len(saved)} file(s) to Drive')

# Restore any files saved from a previous session
restored = []
for fp in glob.glob(f'{DRIVE_RESULTS}/*'):
    dst = os.path.join('results', os.path.basename(fp))
    if not os.path.exists(dst):
        shutil.copy2(fp, dst)
        restored.append(os.path.basename(fp))

if restored:
    print(f'✓ Restored {len(restored)} file(s) from Drive: {restored}')
else:
    print('✓ Drive mounted — no previous results to restore')
print(f'Checkpoint folder: {DRIVE_RESULTS}')

---
## Step 1 — Generate corpus
**~2-4 hours. Fully resumable.**

150 prompts × 3 datasets × 2 (watermarked + control) = 900 completions.  
Output: `results/corpus_llama_d2.jsonl`

If this file already exists from a previous run, the script resumes from where it left off.

In [ ]:
!python scripts/generate_corpus.py --model meta-llama/Llama-3.1-8B-Instruct

GPU: NVIDIA A100-SXM4-40GB
Model:          meta-llama/Llama-3.1-8B-Instruct
Output:         results/corpus_llama_d2.jsonl
δ=2.0  γ=0.5  seed=42  n_per_dataset=150
Loading tokenizer and model: meta-llama/Llama-3.1-8B-Instruct on cuda
config.json: 100% 855/855 [00:00<00:00, 3.74MB/s]
tokenizer_config.json: 55.4kB [00:00, 84.4MB/s]
tokenizer.json: 9.09MB [00:00, 28.0MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.50MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 23.9kB [00:00, 57.9MB/s]
Fetching 4 files: 100% 4/4 [00:40<00:00, 10.12s/it]
Download complete: 100% 16.1G/16.1G [00:40<00:00, 396MB/s]                
Loading weights: 100% 291/291 [00:05<00:00, 55.95it/s, Materializing param=model.norm.weight]                              
generation_config.json: 100% 184/184 [00:00<00:00, 966kB/s]
Loading cnn_dailymail...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cnn_dailymail' isn't based on a loa

In [ ]:
from pipeline.generate import load_corpus
import numpy as np
corpus = load_corpus('results/corpus_llama_d2.jsonl')
wm  = [x for x in corpus if x['watermarked']]
uwm = [x for x in corpus if not x['watermarked']]
print(f'Watermarked:   {len(wm)}')
print(f'Unwatermarked: {len(uwm)}')
print(f'Avg tokens — wm: {np.mean([x["n_tokens"] for x in wm]):.0f},  uwm: {np.mean([x["n_tokens"] for x in uwm]):.0f}')
print(f'Sources: {set(x["source"] for x in corpus)}')
assert len(wm) > 200
print('\n✅  Corpus looks good')

In [ ]:
checkpoint()  # saves corpus_llama_d2.jsonl to Drive

---
## Step 2 — Detection + perplexity eval
**~30 min.**

Output: `results/detection_llama_summary.json` + `results/detection_llama_zscores.npz`  
Acceptance criterion: TPR @ 1% FPR on ≥150-token completions > 0.90

In [ ]:
!python scripts/eval_detection.py --model meta-llama/Llama-3.1-8B-Instruct

In [ ]:
import json
with open('results/detection_llama_summary.json') as f:
    s = json.load(f)
print(f"Calibrated threshold:  {s['calibrated_z_threshold']:.3f}")
print(f"TPR (all lengths):     {s['tpr_at_1pct_fpr_all']:.1%}")
long_key = next((k for k in s if 'ge150' in k), None)
if long_key:
    v = s[long_key]
    status = '✅' if v > 0.90 else '❌'
    print(f"TPR (>=150 tokens):    {v:.1%}  {status}  (need >0.90)")
print(f"PPL ratio (wm/uwm):    {s['ppl_ratio_wm_over_uwm']:.3f}  (ideally close to 1.0)")

In [ ]:
checkpoint()  # saves detection_llama_summary.json + zscores.npz to Drive

---
## Step 3 — Length curves
**~10 min. No model needed.**

Output: `results/length_curves_llama.json`

In [ ]:
!python scripts/eval_length.py --model meta-llama/Llama-3.1-8B-Instruct

In [ ]:
import json
with open('results/length_curves_llama.json') as f:
    lc = json.load(f)
print(f"{'tokens':>8}  {'TPR':>6}")
for row in lc:
    bar = '█' * int(row['tpr'] * 25)
    print(f"{row['n_tokens']:>8}  {row['tpr']:>6.3f}  {bar}")

In [ ]:
checkpoint()  # saves length_curves_llama.json to Drive

---
## Step 4 — Robustness sweep
**~15 min basic / ~90 min with LLM paraphrase.**

Output: `results/robustness_llama.json`

In [ ]:
# Basic: word substitution + token insert/delete (~15 min)
!python scripts/eval_robustness.py --model meta-llama/Llama-3.1-8B-Instruct

# Uncomment to also run LLM paraphrase attack (~90 min):
# !python scripts/eval_robustness.py --model meta-llama/Llama-3.1-8B-Instruct --with-paraphrase

In [ ]:
import json
with open('results/robustness_llama.json') as f:
    rob = json.load(f)
print(f"{'Condition':<32} {'TPR':>6}")
for cond, vals in rob.items():
    bar = '█' * int(vals['tpr_at_1pct_fpr'] * 25)
    print(f"{cond:<32} {vals['tpr_at_1pct_fpr']:>6.1%}  {bar}")

In [ ]:
checkpoint()  # saves robustness_llama.json to Drive

---
## Step 5 — Delta sweep
**~2-3 hours.**

Output: `results/delta_sweep_llama.json`

In [ ]:
!python scripts/eval_delta_sweep.py --model meta-llama/Llama-3.1-8B-Instruct

In [ ]:
import json
with open('results/delta_sweep_llama.json') as f:
    sweep = json.load(f)
print(f"{'delta':>7} {'TPR':>7} {'PPL_wm':>8} {'ratio':>7}")
for row in sweep:
    print(f"{row['delta']:>7.1f} {row['tpr']:>7.3f} {row['mean_ppl_wm']:>8.1f} {row['ppl_ratio']:>7.3f}")

In [ ]:
checkpoint()  # saves delta_sweep_llama.json to Drive

---
## Download results

**Run before closing the session** — Colab deletes `/content/` when the session ends.

Downloads a zip of `results/` to your local machine. Upload this zip to the `colab_combined` notebook when both models are done.

In [ ]:
import zipfile
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
archive   = f'/content/results_llama_{timestamp}.zip'

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in Path('results').rglob('*'):
        if fp.is_file():
            zf.write(fp)

size_mb = Path(archive).stat().st_size / 1e6
print(f'Created {archive}  ({size_mb:.1f} MB)')

from google.colab import files
files.download(archive)